# MicroImpute Pipeline Demo

Demonstrates autoimpute, distribution evaluation, predictor analysis, and dashboard formatting.

## Setup: Import libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.datasets import load_diabetes

from microimpute.comparisons.autoimpute import autoimpute
from microimpute.comparisons.metrics import compare_distributions
from microimpute.visualizations import method_comparison_results
from microimpute.evaluations.predictor_analysis import (
    compute_predictor_correlations,
    leave_one_out_analysis,
    progressive_predictor_inclusion,
)
from microimpute.utils.dashboard_formatter import format_csv
from microimpute.models import OLS

warnings.filterwarnings("ignore")

## Step 1: Load and prepare data

In [2]:
# Load the diabetes dataset
diabetes = load_diabetes()
diabetes_data = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

# Split into donor and receiver portions (70/30 split)
np.random.seed(42)
donor_indices = np.random.choice(
    len(diabetes_data),
    size=int(0.7 * len(diabetes_data)),
    replace=False,
)
receiver_indices = np.array(
    [i for i in range(len(diabetes_data)) if i not in donor_indices]
)

donor_data = diabetes_data.iloc[donor_indices].reset_index(drop=True)
receiver_data = diabetes_data.iloc[receiver_indices].reset_index(drop=True)

In [3]:
# Create a categorical risk_factor variable based on cholesterol levels (s4)
def categorize_risk(s4_value):
    if s4_value < -0.02:
        return "low"
    elif s4_value < 0.02:
        return "medium"
    else:
        return "high"


donor_data["risk_factor"] = donor_data["s4"].apply(categorize_risk)
receiver_data["risk_factor"] = receiver_data["s4"].apply(categorize_risk)

In [4]:
# Define predictors and variables to impute
predictors = ["age", "sex", "bmi", "bp"]
imputed_variables = ["s1", "s4", "risk_factor"]

# Remove imputed variables from receiver data
receiver_data = receiver_data.drop(columns=imputed_variables)

print(f"Donor data shape: {donor_data.shape}")
print(f"Receiver data shape: {receiver_data.shape}")
print(f"Predictors: {predictors}")
print(f"Variables to impute: {imputed_variables}")
print(f"\nRisk factor distribution in donor data:")
print(donor_data["risk_factor"].value_counts())

Donor data shape: (309, 11)
Receiver data shape: (133, 8)
Predictors: ['age', 'sex', 'bmi', 'bp']
Variables to impute: ['s1', 's4', 'risk_factor']

Risk factor distribution in donor data:
risk_factor
low       112
high      108
medium     89
Name: count, dtype: int64


## Step 2: Run autoimpute to find the best imputation method

In [5]:
autoimpute_results = autoimpute(
    donor_data=donor_data,
    receiver_data=receiver_data,
    predictors=predictors,
    imputed_variables=imputed_variables,
    tune_hyperparameters=False,
    impute_all=True,
    k_folds=3,
)

all_attributes = dir(autoimpute_results)
non_dunder_attributes = [
    attr
    for attr in all_attributes
    if (not (attr.startswith("__") and attr.endswith("__")))
]
non_dunder_attributes

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:   19.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    1.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
QuantReg does not support categorical variable 'risk_factor'. Skipping QuantReg for this fold.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.02189326286315918s.) Setting batch_size=2.
QuantReg does not support categorical variable 'risk_factor'. Skipping QuantReg for this fold.
QuantReg does not support categorical variable 'risk_factor'. Skipping QuantReg for this fold.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.0s finished
QuantReg cannot handle the provided variable types. Returning NaN results.
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done   3 out of   3 

['_abc_impl',
 '_calculate_keys',
 '_copy_and_set_values',
 '_get_value',
 '_iter',
 '_setattr_handler',
 'construct',
 'copy',
 'cv_results',
 'dict',
 'fitted_models',
 'from_orm',
 'imputations',
 'json',
 'model_computed_fields',
 'model_config',
 'model_construct',
 'model_copy',
 'model_dump',
 'model_dump_json',
 'model_extra',
 'model_fields',
 'model_fields_set',
 'model_json_schema',
 'model_parametrized_name',
 'model_post_init',
 'model_rebuild',
 'model_validate',
 'model_validate_json',
 'model_validate_strings',
 'parse_file',
 'parse_obj',
 'parse_raw',
 'receiver_data',
 'schema',
 'schema_json',
 'update_forward_refs',
 'validate']

In [6]:
best_method_name = autoimpute_results.fitted_models["best_method"].__class__.__name__
print(f"Best performing method: {best_method_name}")

Best performing method: OLSResults


## Step 3: Compare model performance

In [7]:
# Extract the model cross-validation results from the autoimpute results object
comparison_viz = method_comparison_results(
    data=autoimpute_results.cv_results,
    metric="quantile_loss",
    data_format="wide",
)
fig = comparison_viz.plot(
    title="Autoimpute method comparison on diabetes dataset",
    show_mean=True,
)
fig.show()

## Step 4: Evaluate distribution preservation

In [8]:
distribution_comparison_df = compare_distributions(
    donor_data=donor_data,
    receiver_data=autoimpute_results.receiver_data,
    imputed_variables=imputed_variables,
)

print("Distribution comparison results:")
print(distribution_comparison_df)

Distribution comparison results:
      Variable                Metric  Distance
0           s1  wasserstein_distance  0.024660
1           s4  wasserstein_distance  0.020422
2  risk_factor         kl_divergence  6.033155


## Step 5: Analyze predictor correlations and mutual information

In [9]:
predictor_correlations = compute_predictor_correlations(
    data=donor_data,
    predictors=predictors,
    imputed_variables=imputed_variables,
    method="all",
)

print("Correlation analysis completed:")
print(f"  - Pearson correlation matrix: {predictor_correlations['pearson'].shape}")
print(f"  - Spearman correlation matrix: {predictor_correlations['spearman'].shape}")
print(f"  - Mutual information matrix: {predictor_correlations['mutual_info'].shape}")
print(f"  - Predictor-target MI: {predictor_correlations['predictor_target_mi'].shape}")

Correlation analysis completed:
  - Pearson correlation matrix: (4, 4)
  - Spearman correlation matrix: (4, 4)
  - Mutual information matrix: (4, 4)
  - Predictor-target MI: (4, 3)


## Step 6: Assess predictor importance via leave-one-out analysis

In [10]:
predictor_importance_df = leave_one_out_analysis(
    data=donor_data,
    predictors=predictors,
    imputed_variables=imputed_variables,
    model_class=OLS,
    quantiles=[0.1, 0.5, 0.9],
    train_size=0.7,
    n_jobs=1,
    random_state=42,
)

print("Predictor importance results:")
print(predictor_importance_df[["predictor_removed", "relative_impact"]])

Leave-one-out analysis:   0%|          | 0/4 [00:00<?, ?it/s]

Predictor importance results:
  predictor_removed  relative_impact
1               sex        30.680328
2               bmi         0.032479
0               age         0.004546
3                bp        -0.004827


## Step 7: Analyze impact of variable ordering via progressive inclusion

In [11]:
progressive_results = progressive_predictor_inclusion(
    data=donor_data,
    predictors=predictors,
    imputed_variables=imputed_variables,
    model_class=OLS,
    quantiles=[0.1, 0.5, 0.9],
    train_size=0.7,
    random_state=42,
)

progressive_inclusion_df = progressive_results["results_df"]
optimal_subset = progressive_results["optimal_subset"]
optimal_loss = progressive_results["optimal_loss"]

print(
    f"Optimal predictor order: {progressive_inclusion_df['predictor_added'].tolist()}"
)
print(f"Optimal subset: {optimal_subset}")
print(f"Optimal loss: {optimal_loss:.6f}")

Progressive inclusion:   0%|          | 0/4 [00:00<?, ?it/s]

Optimal predictor order: ['sex', 'bmi', 'age', 'bp']
Optimal subset: ['sex', 'bmi', 'age']
Optimal loss: 2.409709


## Step 8: Format all results into unified CSV for dashboard visualization

In [12]:
output_path = "microimputation_results.csv"

autoimpute_dict = {"cv_results": autoimpute_results.cv_results}

formatted_df = format_csv(
    output_path=output_path,
    autoimpute_result=autoimpute_dict,
    comparison_metrics_df=None,
    distribution_comparison_df=distribution_comparison_df,
    predictor_correlations=predictor_correlations,
    predictor_importance_df=predictor_importance_df,
    progressive_inclusion_df=progressive_inclusion_df,
    best_method_name=best_method_name,
)

print(f"Formatted DataFrame shape: {formatted_df.shape}")
print(f"Result types included: {formatted_df['type'].unique()}")

Formatted DataFrame shape: (49, 9)
Result types included: <ArrowStringArray>
['distribution_distance', 'predictor_correlation',   'predictor_target_mi',
  'predictor_importance', 'progressive_inclusion']
Length: 5, dtype: str


## Step 9: Remember to explore your results in the Microimputation Dashboard!